# High-Volume ADE (DPT-3) → Snowflake — Demo

Parse and extract a batch of invoices with **LandingAI ADE DPT-3**, and stream the
structured results into **Snowflake** via staged `COPY INTO`.

Run the cells top to bottom. Watch **step 5** — the progress bar and the run
summary show the speed.

## 1  ·  Configuration

In [1]:
from config import Settings

S = Settings()
print('✅ Configuration loaded')
print(f'   Parse model    : {S.PARSE_MODEL}')
print(f'   Extract model  : {S.EXTRACT_MODEL}')
print(f'   Workers        : {S.MAX_WORKERS}')
print(f'   Snowflake target: {S.DATABASE}.{S.SNOWFLAKE_SCHEMA}')

✅ Configuration loaded
   Parse model    : dpt-3-pro-latest
   Extract model  : extract-latest
   Workers        : 16
   Snowflake target: DEMOS_ADE_FINANCE.INVOICES_DPT3


## 2  ·  The input documents

A folder of invoices — different vendors, different layouts, no templates.

In [2]:
from run_demo import gather_files

files = gather_files(S.input_dir, S.file_exts)
print(f'📄 Found {len(files)} document(s) in "{S.input_dir}":')
for f in files:
    print('   •', f.split('/')[-1])

📄 Found 27 document(s) in "/Users/avaxia/Desktop/MATERIALS/invoices":
   • invoice_1.pdf
   • invoice_10.pdf
   • invoice_11.pdf
   • invoice_12.pdf
   • invoice_13.pdf
   • invoice_14.pdf
   • invoice_15.pdf
   • invoice_16.pdf
   • invoice_17.pdf
   • invoice_18.pdf
   • invoice_19.pdf
   • invoice_2.pdf
   • invoice_20.pdf
   • invoice_21.pdf
   • invoice_22.pdf
   • invoice_23.pdf
   • invoice_24.pdf
   • invoice_25.pdf
   • invoice_26.pdf
   • invoice_27.pdf
   • invoice_3.pdf
   • invoice_4.pdf
   • invoice_5.pdf
   • invoice_6.pdf
   • invoice_7.pdf
   • invoice_8.PDF
   • invoice_9.pdf


## 3  ·  Canary — parse + extract one invoice

DPT-3 runs two calls: `client.v2.parse` then `client.v2.extract` (schema in,
structured fields out). Let's prove it on a single document first.

In [3]:
from ade_client import build_client, parse_and_extract
from invoice_schema import InvoiceExtractionSchema

client = build_client(S)
sample = files[0]
print(f'🔍 Parsing + extracting: {sample.split("/")[-1]} ...')
pr, er = parse_and_extract(client, sample, InvoiceExtractionSchema, S)

n_blocks = sum(len(p.children or []) for p in (pr.structure.children or []))
print(f'   ✅ {pr.metadata.page_count} page(s) parsed · {n_blocks} blocks found')

ex  = er.extraction
inv = ex.get('invoice_info', {}); co = ex.get('company_info', {}); tot = ex.get('totals_summary', {})
print('\n📋 Extracted fields:')
print(f'   Invoice #   : {inv.get("invoice_number")}')
print(f'   Date        : {inv.get("invoice_date")}')
print(f'   Supplier    : {co.get("supplier_name")}')
print(f'   Total due   : {tot.get("total_due")} {tot.get("currency") or ""}')
print(f'   Line items  : {len(ex.get("line_items", []))}')

🔍 Parsing + extracting: invoice_1.pdf ...
   ✅ 1 page(s) parsed · 9 blocks found

📋 Extracted fields:
   Invoice #   : INV33543191
   Date        : 2020-07-29
   Supplier    : Zoom Video Communications Inc.
   Total due   : 149.9 USD
   Line items  : 1


## 4  ·  Connect to Snowflake

Opens **one** connection for the whole run. The first time, a browser window
pops up for SSO sign-in (it caches after that).

In [4]:
from sf_loader import sf_connect, ensure_formats_and_stages

conn = sf_connect(S)                 # ← browser SSO login happens here (once)
ensure_formats_and_stages(S, conn)   # create ingest stage + file formats
print('✅ Connected to Snowflake · stages + file formats ready')

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://accounts.google.com/o/saml2/idp?idpid=C03ya9g2w&SAMLRequest=jZJfb5swFMW%2FCvKegw1pM2qFRJQsW9Z0ZQ2ptr454FCrxqa%2BpiTffiZ%2FpO6hVR%2BQkH2Of%2Ffec8fTXS29V25AaBWjwCfI46rQpVBVjNb5fBAhDyxTJZNa8RjtOaDpZAyslg1NWvuk7vlLy8F67iEFtL%2BIUWsU1QwEUMVqDtQWdJXcLmnoE8oAuLEOh06WEoRjPVnbUIy7rvO7oa9NhUNCCCZX2Kl6yRf0BtF8zGiMtrrQ8mzZuZ7eQQSYXPQIp3CE7GS8Fuo4go8om6MI6I88zwbZ3SpHXnLuLtUK2pqbFTevouDr%2B%2BWxAHAVrNLr6DKIRj4o3W0le%2BaFrpvWusd894e3vMRSV8KNaDGLUfMsyps%2FOvrJt7eQJqP8%2BzoL1y%2Bb2ezbvC2Wm9Xm918bwfBu%2B2jy3bpA3sM50LAPdAHQ8oXqY7TuiISjAYkGQZCHQxqE9DLyycXXR%2BTNXIxCMXtwnmtlRaFbZcGvtK4kP9SncZ9BiEXZTN0nyjglwz27qsIOHXeDHphm8smOx%2Fit6bRdv9zAF7NMS1Hsvbk2NbPv5xH4weFElIPtQUp5zYRMytJwAJeLlLpLDWfWLbE1LUd4cqT%2Bv8aTfw%3D%3D&RelayState=ver%3A3-hint%3A28518063158538246-ETMsDgAAAZ%2FzGby%2FABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEPJ0Bufay4ek6CrV%2Fir5VCUAAACgoDMOTlVDHs3qmqrKRj7uMQYf1YkBzbKx9CSDmC8OlanjH6Z7v

## 5  ·  Stream the batch into Snowflake  🚀

Documents are parsed + extracted **concurrently**, and each one's rows are staged
and `COPY`ed the moment it finishes — so rows land continuously. Watch the
**progress bar** (docs/s, rows→Snowflake) and the **Concurrency** line in the
summary: that's the speed story.

*Tip: open Snowsight and run `SELECT COUNT(*) FROM INVOICES_MAIN;` while this runs.*

In [5]:
from pipeline import run_streaming

metrics = run_streaming(files, InvoiceExtractionSchema, S, conn=conn)
print(metrics.summary())

Parsing + extracting → Snowflake:   0%|          | 0/27 [00:00<?, ?doc/s]


==================== Run summary ====================
  Documents:        27 ok, 0 failed (27 submitted)
  Pages:            34
  Rows -> Snowflake:494
  Wall time:        73.4s
  Throughput:       0.4 docs/s, 0.5 pages/s, 7 rows/s
  Avg parse+extract:30.71s/doc
  Concurrency:      829s of parse+extract done in 73s wall  ->  11.3x overlap
  (point --input at a larger folder to scale; the pattern streams the
   same way whether it's 4 documents or 4,000)


## 6  ·  Verify — the four tables + the originals stage

In [6]:
from run_demo import verify_counts

verify_counts(S, conn=conn)


Row counts in Snowflake:
  INVOICES_MAIN                  27
  INVOICE_LINE_ITEMS             79
  PARSED_BLOCKS                 361
  MARKDOWN                       27
  RAW_DOCS (stage)               27  original file(s)


## 7  ·  Query the structured results

Plain SQL from here — headers joined to line items.

In [7]:
from sf_loader import fq_table

query = f'''
SELECT m.supplier_name, m.invoice_number, m.total_due,
       COUNT(li.line_index) AS line_items
FROM {fq_table(S, S.table_main)} m
LEFT JOIN {fq_table(S, S.table_lines)} li USING (invoice_uuid)
GROUP BY 1, 2, 3
ORDER BY m.total_due DESC NULLS LAST
'''
cur = conn.cursor(); cur.execute(query)
print(f'{"SUPPLIER":<34} {"INVOICE #":<16} {"TOTAL":>14}  LINES')
print('-' * 74)
for supplier, number, total, lines in cur.fetchall():
    total_str = f'{float(total):,.2f}' if total is not None else '—'
    print(f'{(supplier or "")[:33]:<34} {str(number):<16} {total_str:>14}  {lines}')

SUPPLIER                           INVOICE #             TOTAL  LINES
------------------------------------------------------------------------


TypeError: float() argument must be a string or a real number, not 'NoneType'

## Done

Close the connection. To scale up, point `gather_files` at a folder of your own PDFs — the pattern streams the same way whether it's 4 documents or 4,000.

In [ ]:
conn.close()
print('✅ Done — connection closed.')